# Day 10 — Seq2Seq & Attention Forecasting | RetailMind AI

**Objective:** Upgrade the Day 9 next-day LSTM into a multi-horizon forecaster that predicts the next 14 days and exposes attention weights over the input history.

### Model design
- **Encoder:** LSTM reads the previous 56 days and returns a hidden representation for every input day.
- **Decoder:** LSTM produces a sequence of 14 future representations.
- **Attention:** Each decoder step attends over the encoder's 56 hidden states.
- **Output:** 14 daily sales forecasts.

### Dataset and project layout
Use the same Rossmann `train.csv` as Day 9:

```text
RetailMind/
├── data/rossmann/train.csv
├── models/demand_lstm.keras          # Day 9 model
├── models/demand_scalers.pkl         # Day 9 per-store scales
├── models/
├── notebooks/day10_seq2seq_attention.ipynb
└── src/
```

This notebook uses TensorFlow/Keras for model building and training. Python standard-library modules handle CSV/date parsing and pickle loading. No pandas or NumPy preprocessing is required.

> Run from `RetailMind/notebooks/`, or update `ROOT_DIR` below.

## 1. Imports and configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
from collections import defaultdict
import csv
import pickle

import tensorflow as tf
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)

ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

DATA_PATH = ROOT_DIR / "data" / "rossmann" / "train.csv"

MODEL_DIR = ROOT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DAY9_MODEL_PATH = MODEL_DIR / "demand_lstm.keras"
DAY9_METADATA_PATH = MODEL_DIR / "demand_scalers.pkl"

SEQ2SEQ_MODEL_PATH = MODEL_DIR / "demand_seq2seq_attention.keras"
ATTENTION_MODEL_PATH = MODEL_DIR / "demand_attention_inspector.keras"
METADATA_PATH = MODEL_DIR / "demand_seq2seq_metadata.pkl"
PREDICTIONS_PATH = ROOT_DIR / "data" / "demand_14day_predictions.csv"

LOOKBACK = 56
HORIZON = 14
TEST_DAYS = 42
EPOCHS = 30
BATCH_SIZE = 256
RANDOM_SEED = 42

tf.keras.utils.set_random_seed(RANDOM_SEED)

print("Project root:", ROOT_DIR)
print("Dataset exists:", DATA_PATH.exists())
print("Day 9 model exists:", DAY9_MODEL_PATH.exists())
print("Day 9 metadata exists:", DAY9_METADATA_PATH.exists())

## 2. Load Rossmann data using Python's standard library

We aggregate records by store and date. This keeps preprocessing within the TensorFlow/Keras + standard-library constraint.

The Rossmann `Sales` column is used as the target. The model is a sales-history-only baseline; known future promotions and holidays are not included yet.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}. Put Rossmann train.csv in data/rossmann/."
    )

sales_by_store = defaultdict(dict)

with DATA_PATH.open("r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    required = {"Store", "Date", "Sales"}
    if not required.issubset(reader.fieldnames or []):
        raise ValueError(f"CSV must contain columns: {required}")

    for row in reader:
        store_id = int(row["Store"])
        date = datetime.strptime(row["Date"], "%Y-%m-%d").date()
        sales = float(row["Sales"] or 0)
        sales_by_store[store_id][date] = (
            sales_by_store[store_id].get(date, 0.0) + sales
        )

print("Stores loaded:", len(sales_by_store))
first_store = sorted(sales_by_store)[0]
print("Example store:", first_store)
print("Example rows:", list(sorted(sales_by_store[first_store].items()))[:5])

## 3. Create continuous daily series per store

We create a continuous daily calendar between each store's first and last recorded date. Missing calendar dates are filled with zero sales, matching the simplifying assumption used in Day 9.

In [ ]:
store_dates = {}
store_values = {}

for store_id, date_sales in sales_by_store.items():
    first_date = min(date_sales)
    last_date = max(date_sales)
    number_of_days = (last_date - first_date).days + 1

    dates = [first_date + timedelta(days=i) for i in range(number_of_days)]
    values = [float(date_sales.get(day, 0.0)) for day in dates]

    store_dates[store_id] = dates
    store_values[store_id] = tf.constant(values, dtype=tf.float32)

print("Continuous series prepared:", len(store_values))
print("Example series length:", len(store_values[first_store]))

## 4. Build sequence-to-sequence windows

For each training example:
- Encoder input: previous **56 days**.
- Decoder target: next **14 days**.
- The final **42 days** per store are held out for testing.
- Scaling uses only the pre-test training period maximum for that store.

The training windows are restricted so their entire 14-day target lies before the test boundary. Test windows begin at the test boundary and roll forward by one day; each test input uses the preceding 56 actual observations.

In [ ]:
def create_seq2seq_windows(
    values_by_store,
    dates_by_store,
    lookback=LOOKBACK,
    horizon=HORIZON,
    test_days=TEST_DAYS
):
    X_train_parts, y_train_parts = [], []
    X_test_parts, y_test_parts = [], []
    test_dates, test_stores = [], []
    scalers = {}

    for store_id, tensor_values in values_by_store.items():
        values = tensor_values.numpy()
        dates = dates_by_store[store_id]

        split_idx = len(values) - test_days
        if split_idx <= lookback + horizon:
            continue

        scale = float(tf.reduce_max(tensor_values[:split_idx]).numpy())
        if not tf.math.is_finite(scale) or scale <= 0:
            scale = 1.0
        scalers[int(store_id)] = scale

        normalized = values / scale

        # Window start s predicts target indices [s+lookback, ..., s+lookback+horizon-1].
        last_train_start = split_idx - lookback - horizon
        train_starts = range(0, last_train_start + 1)

        # Test forecast origins are split_idx, split_idx+1, ... where a full
        # 14-day target remains inside the observed test period.
        test_origin_starts = range(
            split_idx - lookback,
            len(values) - lookback - horizon + 1
        )

        train_x, train_y = [], []
        for start in train_starts:
            input_start = start
            target_start = start + lookback
            train_x.append(normalized[input_start:target_start])
            train_y.append(normalized[target_start:target_start + horizon])

        test_x, test_y = [], []
        for start in test_origin_starts:
            input_start = start
            target_start = start + lookback
            test_x.append(normalized[input_start:target_start])
            test_y.append(normalized[target_start:target_start + horizon])
            test_dates.append(dates[target_start:target_start + horizon])
            test_stores.append(int(store_id))

        if train_x and test_x:
            X_train_parts.append(tf.constant(train_x, dtype=tf.float32)[..., tf.newaxis])
            y_train_parts.append(tf.constant(train_y, dtype=tf.float32)[..., tf.newaxis])
            X_test_parts.append(tf.constant(test_x, dtype=tf.float32)[..., tf.newaxis])
            y_test_parts.append(tf.constant(test_y, dtype=tf.float32)[..., tf.newaxis])

    if not X_train_parts or not X_test_parts:
        raise ValueError("No valid sequence windows. Check dataset length and split settings.")

    X_train = tf.concat(X_train_parts, axis=0)
    y_train = tf.concat(y_train_parts, axis=0)
    X_test = tf.concat(X_test_parts, axis=0)
    y_test = tf.concat(y_test_parts, axis=0)

    return (
        X_train, y_train, X_test, y_test,
        scalers, test_dates, test_stores
    )


(
    X_train, y_train, X_test, y_test,
    scalers, test_dates, test_stores
) = create_seq2seq_windows(store_values, store_dates)

print("X_train:", X_train.shape)  # (samples, 56, 1)
print("y_train:", y_train.shape)  # (samples, 14, 1)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)
print("Stores with usable windows:", len(scalers))

## 5. Build the encoder-decoder with attention

The encoder returns all hidden states, not just its final state. The decoder generates one hidden representation per future day. Keras `Attention` computes a relevance score between each decoder step (query) and every encoder time step (value).

We expose a second inference model that returns both forecasts and attention scores, while the training model returns forecasts only.

In [ ]:
def build_seq2seq_attention(lookback=LOOKBACK, horizon=HORIZON):
    encoder_inputs = tf.keras.layers.Input(
        shape=(lookback, 1), name="history"
    )

    encoder_outputs, state_h, state_c = tf.keras.layers.LSTM(
        64,
        return_sequences=True,
        return_state=True,
        name="encoder_lstm"
    )(encoder_inputs)

    # Seed each decoder time step with the encoder's final hidden state.
    decoder_inputs = tf.keras.layers.RepeatVector(
        horizon, name="repeat_context"
    )(state_h)

    decoder_outputs = tf.keras.layers.LSTM(
        64,
        return_sequences=True,
        name="decoder_lstm"
    )(decoder_inputs, initial_state=[state_h, state_c])

    attention_layer = tf.keras.layers.Attention(
        use_scale=True, name="temporal_attention"
    )

    # Query = decoder outputs; Value/Key = encoder outputs.
    context, attention_scores = attention_layer(
        [decoder_outputs, encoder_outputs],
        return_attention_scores=True
    )

    combined = tf.keras.layers.Concatenate(
        axis=-1, name="context_decoder_concat"
    )([decoder_outputs, context])

    forecast = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(1), name="sales_forecast"
    )(combined)

    training_model = tf.keras.Model(
        inputs=encoder_inputs,
        outputs=forecast,
        name="retailmind_seq2seq_attention"
    )

    attention_model = tf.keras.Model(
        inputs=encoder_inputs,
        outputs=[forecast, attention_scores],
        name="retailmind_attention_inspector"
    )

    training_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="mse",
        metrics=["mae"]
    )

    return training_model, attention_model


model, attention_model = build_seq2seq_attention()
model.summary()

## 6. Train the Seq2Seq model

We use a validation split from the training windows and early stopping. The held-out recent test windows are not passed to `fit`.

Because the samples contain overlapping windows, the validation split is used for training control, not as a fully independent time-based validation benchmark.

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
history_values = history.history
plt.figure(figsize=(10, 5))
plt.plot(history_values["loss"], label="Training loss")
plt.plot(history_values["val_loss"], label="Validation loss")
plt.title("Seq2Seq Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Evaluate 14-day forecasts in original sales units

We report:
- **Overall MAE/RMSE** across all stores, forecast origins, and horizons.
- **Horizon-wise MAE/RMSE** to see whether accuracy degrades farther into the future.

Each test forecast uses the preceding 56 actual sales observations. This is a rolling-origin evaluation, not one uninterrupted 42-day recursive forecast.

In [ ]:
predicted_normalized = model.predict(
    X_test, batch_size=BATCH_SIZE, verbose=0
)

actual_original = tf.zeros_like(y_test)
predicted_original = tf.zeros_like(predicted_normalized)

# Restore each sample using its store-specific training scale.
actual_rows = []
predicted_rows = []

for sample_index, store_id in enumerate(test_stores):
    scale = scalers[int(store_id)]
    actual_rows.append(y_test[sample_index] * scale)
    predicted_rows.append(predicted_normalized[sample_index] * scale)

actual_original = tf.stack(actual_rows)
predicted_original = tf.stack(predicted_rows)

overall_mae = tf.reduce_mean(tf.abs(actual_original - predicted_original))
overall_rmse = tf.sqrt(tf.reduce_mean(tf.square(actual_original - predicted_original)))

print(f"Overall 14-day MAE : {overall_mae.numpy():.2f}")
print(f"Overall 14-day RMSE: {overall_rmse.numpy():.2f}")

horizon_metrics = []
for step in range(HORIZON):
    step_actual = actual_original[:, step, 0]
    step_pred = predicted_original[:, step, 0]
    step_mae = tf.reduce_mean(tf.abs(step_actual - step_pred))
    step_rmse = tf.sqrt(tf.reduce_mean(tf.square(step_actual - step_pred)))
    horizon_metrics.append((step + 1, float(step_mae.numpy()), float(step_rmse.numpy())))

print("\nHorizon | MAE | RMSE")
for horizon_day, step_mae, step_rmse in horizon_metrics:
    print(f"{horizon_day:>7} | {step_mae:>10.2f} | {step_rmse:>10.2f}")

## 8. Compare against the Day 9 one-step LSTM

For a fair multi-day comparison, we roll the Day 9 one-step model forward **autoregressively**: after predicting day 1, that prediction is appended to the input window to predict day 2, and so on. This avoids giving the one-step model actual future sales as inputs.

The comparison uses the same test forecast origins and 14-day targets as the Seq2Seq model. Day 9's model and scaler metadata must exist in `models/`.

In [ ]:
def autoregressive_day9_forecast(
    day9_model,
    history_window,
    scale,
    horizon=HORIZON
):
    # history_window is already normalized and shaped (lookback, 1).
    rolling = tf.identity(history_window)
    forecasts = []

    for _ in range(horizon):
        next_value = day9_model.predict(
            rolling[tf.newaxis, ...], verbose=0
        )[0, 0]
        forecasts.append(next_value)

        rolling = tf.concat(
            [rolling[1:], tf.reshape(next_value, (1, 1))],
            axis=0
        )

    return tf.stack(forecasts) * scale


day9_available = DAY9_MODEL_PATH.exists() and DAY9_METADATA_PATH.exists()

if day9_available:
    day9_model = tf.keras.models.load_model(DAY9_MODEL_PATH)
    with DAY9_METADATA_PATH.open("rb") as file:
        day9_metadata = pickle.load(file)

    day9_scalers = day9_metadata["scalers"]

    baseline_predictions = []
    for sample_index, store_id in enumerate(test_stores):
        if int(store_id) not in day9_scalers:
            raise KeyError(
                f"Store {store_id} is missing from Day 9 scaler metadata."
            )

        # Use Day 9's own scale for its model input.
        day9_scale = float(day9_scalers[int(store_id)])

        # X_test is normalized with Day 10 scale, so recover original values
        # then normalize with the Day 9 scale.
        history_original = X_test[sample_index, :, 0] * scalers[int(store_id)]
        history_day9_scaled = history_original / day9_scale
        baseline_predictions.append(
            autoregressive_day9_forecast(
                day9_model,
                history_day9_scaled[:, tf.newaxis],
                day9_scale
            )
        )

    baseline_predictions = tf.stack(baseline_predictions)[:, :, tf.newaxis]

    baseline_mae = tf.reduce_mean(
        tf.abs(actual_original - baseline_predictions)
    )
    baseline_rmse = tf.sqrt(
        tf.reduce_mean(tf.square(actual_original - baseline_predictions))
    )

    print("Day 9 autoregressive 14-day baseline")
    print(f"MAE : {baseline_mae.numpy():.2f}")
    print(f"RMSE: {baseline_rmse.numpy():.2f}")

    print("\nDay 10 Seq2Seq + Attention")
    print(f"MAE : {overall_mae.numpy():.2f}")
    print(f"RMSE: {overall_rmse.numpy():.2f}")
else:
    print("Day 9 model or scaler metadata not found.")
    print("Skipping baseline comparison. Run the Day 9 notebook and save its artifacts first.")

## 9. Visualize a 14-day forecast

The chart shows one store and one test forecast origin. It compares the 14 actual future days with the Seq2Seq prediction and, when available, the autoregressive Day 9 baseline.

In [ ]:
sample_index = 0
sample_store = int(test_stores[sample_index])
sample_dates = test_dates[sample_index]

plt.figure(figsize=(13, 5))
plt.plot(sample_dates, actual_original[sample_index, :, 0].numpy(),
         marker="o", label="Actual")
plt.plot(sample_dates, predicted_original[sample_index, :, 0].numpy(),
         marker="o", label="Day 10 Seq2Seq + Attention")

if day9_available:
    plt.plot(sample_dates, baseline_predictions[sample_index, :, 0].numpy(),
             marker="o", linestyle="--", label="Day 9 autoregressive LSTM")

plt.title(f"14-Day Demand Forecast — Store {sample_store}")
plt.xlabel("Date")
plt.ylabel("Daily sales")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

## 10. Visualize attention weights

The attention score matrix has shape `(horizon, lookback)` for one sample:
- Rows correspond to forecast days 1–14.
- Columns correspond to historical input days 1–56.
- Each row shows how strongly that forecast step attends to each encoder time step.

Attention weights are a model diagnostic: they show where the network places weight, but should not be treated as definitive causal explanations.

In [ ]:
sample_attention_input = X_test[sample_index:sample_index + 1]

sample_forecast, sample_attention = attention_model.predict(
    sample_attention_input, verbose=0
)

# Expected shape: (1, horizon, lookback)
attention_matrix = sample_attention[0]

plt.figure(figsize=(14, 6))
plt.imshow(attention_matrix, aspect="auto", cmap="viridis")
plt.colorbar(label="Attention weight")
plt.xlabel("Historical day (oldest → most recent)")
plt.ylabel("Forecast horizon day")
plt.yticks(range(HORIZON), [f"Day {i}" for i in range(1, HORIZON + 1)])
plt.xticks(
    ticks=range(0, LOOKBACK, 5),
    labels=[str(i + 1) for i in range(0, LOOKBACK, 5)]
)
plt.title(f"Attention Weights — Store {sample_store}")
plt.tight_layout()
plt.show()

print("Attention matrix shape:", attention_matrix.shape)
print("Attention row sums (should be close to 1):")
print(tf.reduce_sum(sample_attention[0], axis=-1).numpy())

## 11. Save the upgraded demand forecaster

We save:
- The forecast model.
- The attention-inspection model.
- Per-store scaling metadata and sequence configuration.
- The multi-horizon predictions.

The metadata is needed to normalize inputs consistently during inference.

In [ ]:
model.save(SEQ2SEQ_MODEL_PATH)
attention_model.save(ATTENTION_MODEL_PATH)

metadata = {
    "scalers": scalers,
    "lookback": LOOKBACK,
    "horizon": HORIZON,
    "frequency": "daily",
    "target": "Sales",
    "model_type": "seq2seq_lstm_with_temporal_attention"
}

with METADATA_PATH.open("wb") as file:
    pickle.dump(metadata, file)

# Flatten sample/horizon predictions into a row-per-store/date file.
prediction_rows = []
for sample_index, store_id in enumerate(test_stores):
    for horizon_index, forecast_date in enumerate(test_dates[sample_index]):
        prediction_rows.append({
            "Store": int(store_id),
            "ForecastOrigin": str(test_dates[sample_index][0] - timedelta(days=1)),
            "Date": str(forecast_date),
            "HorizonDay": horizon_index + 1,
            "Actual": float(actual_original[sample_index, horizon_index, 0].numpy()),
            "Predicted": float(predicted_original[sample_index, horizon_index, 0].numpy())
        })

with PREDICTIONS_PATH.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=prediction_rows[0].keys())
    writer.writeheader()
    writer.writerows(prediction_rows)

print("Saved forecast model:", SEQ2SEQ_MODEL_PATH)
print("Saved attention model:", ATTENTION_MODEL_PATH)
print("Saved metadata:", METADATA_PATH)
print("Saved predictions:", PREDICTIONS_PATH)

## 12. Reload and verify inference

Load the saved forecast model and metadata. This final check verifies that the persisted model can generate a 14-day forecast.

In [ ]:
reloaded_model = tf.keras.models.load_model(SEQ2SEQ_MODEL_PATH)

with METADATA_PATH.open("rb") as file:
    reloaded_metadata = pickle.load(file)

reloaded_prediction = reloaded_model.predict(
    X_test[:1], verbose=0
)[0, :, 0]

store_id = int(test_stores[0])
store_scale = float(reloaded_metadata["scalers"][store_id])
forecast_in_sales_units = reloaded_prediction * store_scale

print("Reload successful.")
print("Forecast horizon:", len(forecast_in_sales_units), "days")
print("Next 14 daily sales forecasts:")
for day_number, value in enumerate(forecast_in_sales_units, start=1):
    print(f"Day {day_number:2d}: {float(value):.2f}")

## 13. Day 10 completion checklist

- [ ] Rossmann data loaded and daily store series prepared.
- [ ] 56-day encoder / 14-day decoder windows created.
- [ ] Seq2Seq LSTM with temporal attention trained.
- [ ] Overall and horizon-wise MAE/RMSE calculated.
- [ ] Day 9 autoregressive baseline compared (if Day 9 artifacts exist).
- [ ] Forecast plot inspected.
- [ ] Attention heatmap inspected.
- [ ] Model, metadata, and predictions saved.

### Important interpretation notes
1. The model forecasts **sales**, which are an imperfect proxy for unconstrained demand.
2. Attention weights are diagnostic, not causal explanations.
3. The test metric pools overlapping rolling forecast origins; these are not independent observations.
4. The 14-day forecasts use actual historical observations at each rolling origin. They are not one single 42-day forecast made recursively.
5. The model uses no future-known covariates. Promotions, holidays, and store closures should be added in a future feature-aware model.

### Git upload

From the RetailMind project root:

```bash
git add notebooks/day10_seq2seq_attention.ipynb
git add models/demand_seq2seq_attention.keras
git add models/demand_attention_inspector.keras
git add models/demand_seq2seq_metadata.pkl
git add data/demand_14day_predictions.csv
git commit -m "feat: add seq2seq attention demand forecasting"
git push origin main
```

Avoid committing large datasets or model artifacts if your repository's `.gitignore` / Git LFS policy excludes them.